In [1]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tqdm.pandas()

In [4]:
# Load dataset
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/storage/labeledverses.csv", encoding='utf-8-sig')
print("Initial shape:", df.shape)

Initial shape: (51815, 15)


In [5]:
# Load model
lang_detector = pipeline("text-classification", 
                         model="papluca/xlm-roberta-base-language-detection")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2747.31it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: papluca/xlm-roberta-base-language-detection
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Load model
lang_detector = pipeline("text-classification", 
                         model="papluca/xlm-roberta-base-language-detection")

In [7]:
# Function to detect language and confidence
def detect_language(text):
    try:
        if not isinstance(text, str) or text.strip() == "":
            return "unknown", 0.0
        
        result = lang_detector(text[:512])  # truncate long text
        label = result[0]['label']
        score = result[0]['score']
        
        return label, score
    
    except:
        return "unknown", 0.0

df[['verse_language', 'verse_confidence']] = df['genius_lyrics'].progress_apply(
    lambda x: pd.Series(detect_language(x))
)

100%|██████████| 51815/51815 [29:17<00:00, 29.49it/s]  


In [ ]:
# Reduce decimal places for verse confidence
df['verse_confidence'] = df['verse_confidence'].round(2)

In [10]:
# Filter english verses
# Keep unknown as well, for missing lyrics
eng_df = df[(df["verse_language"] == "en") | (df["verse_language"] == "unknown")]
print(f"Size of English verses: {eng_df.shape[0]}")
eng_df.to_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/englishverses.csv", index=False, encoding="utf-8-sig")

# Filter non-english verses
noeng_df = df[~df.index.isin(eng_df.index)]
print(f"Size of non-English verses: {noeng_df.shape[0]}")
noeng_df.to_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/nonenglishverses.csv", index=False, encoding="utf-8-sig")

Size of English verses: 44915
Size of non-English verses: 6900
